# AXI test 7: ruled-out causes (Gemini 3.5 Flash Lite, free tier)
A model hunts for the real cause of an outage among 240 causes (30 parts, each failing in 8 ways). The host says only NOT or FOUND. The question: when the list of dead causes is kept by the host and shown short, does the model stop proposing a dead cause again, compared with the same facts kept in the chat, and with the chat shortened into a summary every 20 checks (the way an agent tool compacts a long chat)? In words mode the model names a cause in a plain sentence and the host reads the words. The prediction and the reading rules were written before this run in `docs/RULED_OUT_TEST.md`.

**What to know**
* **No GPU.** Use the free CPU runtime (Runtime menu -> *Change runtime type* -> CPU). Nothing is downloaded.
* **A daily limit, and two days.** The free key allows about 480 calls a day. Keys mode needs at most 726 calls and words mode at most 180 (906 in all), so it takes about two days. When it stops, everything finished is saved on your Google Drive: run **Cells 2 to 5 again the next day**, then the cell that stopped.
* **Your key** lives in Colab Secrets (Cell 1). It is never printed or saved in any file. Never paste it in the chat.

Run each cell **one at a time, top to bottom**, wait for the green tick before the next one.

### Cell 1: put your key into Colab Secrets (once; no code to run)
1. In Google AI Studio (`aistudio.google.com`), open **API Keys** in the left menu and copy your key.
2. Here in Colab, tap the **key icon** in the left bar (*Secrets*) -> **Add new secret**.
3. Name: `GEMINI_API_KEY` (exactly). Value: paste the key. Switch **Notebook access** on (it is per notebook, so do it again for this one).
Do not paste the key into a cell or into the chat.

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-ruled-out.zip`

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the Gemini library (about 20 seconds)

In [ ]:
!pip -q install -U google-genai

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). This is where your progress and every reply are kept

In [ ]:
import sys, os
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_ruled_out")

### Cell 5: connect to the model and check it with 6 calls (about 1 minute). It prints SMOKE OK or STOP. If it says STOP, do not run the next cells: send me the printout

In [ ]:
import json, time
from google.colab import userdata
from google import genai
from axi.experiments import api_llm as API
from axi.experiments import gemini_long as G
from axi.experiments import ruled_out as R
from axi.experiments.api_llm import QuotaStop, ServiceBusy

try: KEY = userdata.get("GEMINI_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: add a Secret named GEMINI_API_KEY and switch Notebook access on.")
WANTED = "gemini-3.5-flash-lite"
CACHE = "axi_ruled_out_cache.jsonl"
print("reply file:", store.restore(CACHE))
client = genai.Client(api_key=KEY)
MODEL = API.resolve_model(API.list_model_names(client), WANTED); print("model id:", MODEL)
backend = G.GeminiChatBackend(model=MODEL, thinking=None, temperature=None, max_output_tokens=G.MAX_OUT, client=client)
llm = G.ChatLLM(backend, store.path(CACHE), backend.config_dict(), daily_limit=480, min_interval=4.5, on_write=lambda: store.persist(CACHE))
RESET = "The free quota resets at midnight Pacific time: 12:30 pm in India until 1 November, 1:30 pm after. Then open this notebook again, run Cells 2 to 5, and run the cell that stopped again."
BUSY = "This is Google's server being busy. Wait at least 30 minutes, then run THIS SAME CELL once (if this page is still open, you do not need the earlier cells). Do NOT run it again and again: Google counts failed tries against the free daily limit."
T0 = time.time()
def prog(i, n): print(f"  {i} calls saved   {time.time() - T0:6.0f} s   calls today: {llm.calls_today()}" + ("" if store.persistent else "   (NOT saved to Drive)"), end="\r")
def save(res, path):
    json.dump(res, open(path, "w"), indent=1, default=str); return path
def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e)); print(BUSY if isinstance(e, ServiceBusy) else RESET); return None
res_keys = res_words = None
smoke = go(lambda: R.run_arm(llm, "kept", "keys", R.SEEDS[0], "/content/smoke_ruled.jsonl", 0, meta=dict(model=MODEL), steps=6))
if smoke:
    for r in smoke["rows"]: print(r["step"], repr(r["text"]), r["status"])
    ok = len(smoke["rows"]) == 6 and all(r["status"] == "ok" for r in smoke["rows"])
    print("SMOKE OK" if ok else "STOP: some replies were not read as checks. Do not run the next cells: send me the printout.")

### Cell 6: keys mode. Four arms, three episodes each (at most 726 calls; it stops at the daily limit and carries on the next day). Prints the table and the eight reading-rule lines at the end

In [ ]:
T0 = time.time()
name = "axi_ruled_out_keys"
print("progress file:", store.restore(name + ".jsonl"))
res_keys = go(lambda: R.run_mode(llm, "keys", store.path(name + ".jsonl"), meta=dict(model=MODEL), progress=store.progress(name + ".jsonl", prog)))
if res_keys:
    print(); R.print_summary(res_keys["summary"], "keys", MODEL)
    R.print_verdict(R.verdict_keys(res_keys["summary"]), "keys mode")
    sm = [r for r in res_keys["rows"] if r["kind"] == "summary"]
    for r in sm[:1] + sm[-1:]: print(f"\nthe model's summary after reply {r['step']} of episode {r['ep']} (cut at 700 characters):\n  " + r["text"][:700])
    print("\ncalls today:", llm.calls_today())
    files.download(save(res_keys, store.path(name + ".json"))); store.persist(name + ".json")

### Cell 7: words mode. Two arms, three episodes each (at most 180 calls). Prints the table and the three reading-rule lines at the end

In [ ]:
assert res_keys, "The cell above stopped for today. Run it again first."
T0 = time.time()
name = "axi_ruled_out_words"
print("progress file:", store.restore(name + ".jsonl"))
res_words = go(lambda: R.run_mode(llm, "words", store.path(name + ".jsonl"), meta=dict(model=MODEL), progress=store.progress(name + ".jsonl", prog)))
if res_words:
    print(); R.print_summary(res_words["summary"], "words", MODEL)
    R.print_verdict(R.verdict_words(res_words["summary"]), "words mode")
    print("totals:", llm.usage())
    files.download(save(res_words, store.path(name + ".json"))); store.persist(name + ".json")
    files.download(store.path(CACHE))

### Done
Files downloaded: `axi_ruled_out_keys.json`, `axi_ruled_out_words.json` and `axi_ruled_out_cache.jsonl`. Upload them to the chat, with a screenshot of the printed tables and the 'reading rule' lines. Do not paste any keys anywhere.